# Detección de melanomas — entrenamiento conjunto (VGG16 · ResNet50V2 · EfficientNetV2S)

**Clasificación binaria de lesiones dermatoscópicas (benigno / maligno)**

Adrián Barriuso Pizarro · 2026

---

Notebook único que combina los tres notebooks de `notebooks/` (`vgg16.ipynb`,
`resnet50v2.ipynb`, `efficientnetv2s.ipynb`) para entrenar las tres
arquitecturas **de una sola tirada**, en **Kaggle** (mismas rutas e
inicialización que en `resnet50v2-completed.ipynb`), reutilizando el mismo
pipeline de datos y las mismas funciones auxiliares para las tres.

| Arquitectura | Backbone | Preprocesado interno | Fase 2 (fine-tuning) |
|---|---|---|---|
| VGG16 | GAP→Dense(256)→Dropout(0.5)→Dense(1) | Lambda: `preprocess_input(x*255)` (BGR+medias Caffe) | Últimas 4 capas, Adam 1e-5, 30 epochs |
| ResNet50V2 | GAP→Dense(256)→Dropout(0.5)→Dense(1) | `Rescaling(2, offset=-1)` → [-1,1] | ~80 capas no-BN, BN congelado, Adam 1e-6, 40 epochs |
| EfficientNetV2S | GAP→Dense(256)→Dropout(0.5)→Dense(1) | `Rescaling(255)` + `include_preprocessing=True` | 50% capas, BN congelado, Adam 1e-5, 40 epochs |

Las tres comparten: dataset (Melanoma Skin Cancer, 10 000 imágenes, Kaggle,
CC0), tamaño de entrada 224×224, seed 42, Fase 1 con RMSprop 1e-4 / 20 epochs
y `class_weight = {0: 1.0, 1: 1.3}` (penaliza más fallar un maligno).

In [ ]:
# === Setup para Kaggle (GPU T4) ===
# La exportación a TF.js (tensorflowjs) todavía no soporta Keras 3 de forma
# fiable. Forzamos Keras 2 (legacy) en TODO el notebook: la variable debe
# fijarse ANTES de importar TensorFlow. Si Kaggle pide reiniciar el entorno
# tras la instalación, hazlo y reejecuta desde aquí.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [ ]:
import os, random, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    classification_report, roc_curve, auc, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_curve, average_precision_score,
)
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau,
)
from tensorflow.keras.optimizers import RMSprop, Adam
from tensorflow.keras.utils import load_img, img_to_array

# Reproducibilidad
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Comprobación de GPU (en Kaggle: Settings > Accelerator > GPU T4 x2)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        print("GPU disponible:", g.name)
    # Crecimiento de memoria: evita que TF reserve toda la VRAM de golpe
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError as e:
        print(e)
else:
    print("AVISO: no se detectó GPU. El entrenamiento será MUY lento.")
    print("En Kaggle: Settings > Accelerator > GPU T4 x2.")

## 0. Rutas y dataset (Kaggle)

Mismas rutas que `resnet50v2-completed.ipynb`. `MODEL_DIR` es la carpeta
base; cada arquitectura escribe en su propia subcarpeta
`MODEL_DIR/<modelo_id>/`.

In [ ]:
# Kaggle: los datasets adjuntos aparecen bajo /kaggle/input/<slug>/...
# Autodetectamos la carpeta que contiene train/ y test/ para no depender del
# slug exacto con el que hayas adjuntado el dataset.
import glob

def _find_base_dir():
    # 1) Coincidencia directa por nombre habitual
    for cand in glob.glob("/kaggle/input/**/melanoma_cancer_dataset", recursive=True):
        if os.path.isdir(os.path.join(cand, "train")) and os.path.isdir(os.path.join(cand, "test")):
            return cand
    # 2) Cualquier carpeta que contenga train/ y test/ dentro de /kaggle/input
    for train_dir in glob.glob("/kaggle/input/**/train", recursive=True):
        parent = os.path.dirname(train_dir)
        if os.path.isdir(os.path.join(parent, "test")):
            return parent
    return None

BASE_DIR = _find_base_dir()
assert BASE_DIR, (
    "No he encontrado el dataset en /kaggle/input. Añade el dataset "
    "'Melanoma Skin Cancer Dataset of 10000 Images' desde el panel Data del "
    "notebook (icono + Add Input) o revisa que contenga las carpetas train/ y test/."
)
TRAIN_DIR    = os.path.join(BASE_DIR, "train")
TEST_DIR     = os.path.join(BASE_DIR, "test")
TEST_BEN_DIR = os.path.join(TEST_DIR, "benign")
TEST_MAL_DIR = os.path.join(TEST_DIR, "malignant")
CLASS_NAMES  = ["benign", "malignant"]

MODEL_DIR = "/kaggle/working/melanoma_model"

# Hiperparámetros del pipeline (compartidos por los 3 modelos)
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
# Sesgo hacia la sensibilidad: penaliza más fallar un maligno (clase 1)
class_weight = {0: 1.0, 1: 1.3}

assert os.path.isdir(TRAIN_DIR), f"No existe {TRAIN_DIR}."
assert os.path.isdir(TEST_DIR),  f"No existe {TEST_DIR}."
print("Dataset:", BASE_DIR)

## 1. Pipeline de datos con `tf.data` (compartido)

Se construye **una sola vez**: los tres modelos entrenan sobre el mismo
`train_ds` / `val_ds` / `test_ds` (misma seed, mismo split 80/20).
`label_mode='binary'` devuelve etiquetas float32 (0.0/1.0).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary",
)
# shuffle=False en test: las predicciones mantienen el orden de los archivos
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=False,
)

class_names = train_ds.class_names
assert class_names == CLASS_NAMES, f"Orden de clases inesperado: {class_names}"
print("Clases:", class_names)
print(f"Batches  train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")

### Augmentation y normalización

El pipeline entrega imágenes en **[0, 1]**. El preprocesado específico de
cada backbone se hornea dentro del modelo (Lambda/Rescaling), no aquí, para
que cada modelo exportado sea autocontenido. Augmentation geométrica +
fotométrica, solo en entrenamiento.

In [ ]:
normalization = layers.Rescaling(1.0 / 255)

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.15, value_range=(0.0, 1.0)),
    layers.RandomContrast(0.15),
], name="augmentation")

def preprocess_train(x, y):
    x = normalization(x)
    x = augmentation(x, training=True)
    return tf.clip_by_value(x, 0.0, 1.0), y

def preprocess_eval(x, y):
    return normalization(x), y

train_ds = train_ds.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(preprocess_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Smoke test: forzamos que TF materialice un batch para que veas que el pipeline
# funciona (si algo va mal en el augmentation, aquí falla, no dentro de fit()).
print("Comprobando el pipeline...")
_x, _y = next(iter(train_ds))
print(f"  train batch shape: {tuple(_x.shape)} dtype {_x.dtype} | rango [{float(tf.reduce_min(_x)):.3f}, {float(tf.reduce_max(_x)):.3f}]")
print(f"  labels shape:      {tuple(_y.shape)} dtype {_y.dtype}")
print("Pipeline listo.")

## 2. Funciones auxiliares (compartidas)

`crear_callbacks` y `graficar_historial` ahora reciben `modelo_id` para
guardar cada checkpoint en su propia subcarpeta.

In [ ]:
def crear_callbacks(modelo_id, nombre_archivo, paciencia_es=7, paciencia_lr=3):
    """ModelCheckpoint (mejor val_loss) + EarlyStopping + ReduceLROnPlateau."""
    carpeta = os.path.join(MODEL_DIR, modelo_id)
    os.makedirs(carpeta, exist_ok=True)
    ruta = os.path.join(carpeta, nombre_archivo)
    return [
        ModelCheckpoint(ruta, monitor="val_loss", save_best_only=True,
                        mode="min", verbose=1),
        EarlyStopping(monitor="val_loss", patience=paciencia_es,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=paciencia_lr,
                          min_lr=1e-7, verbose=1),
    ]

def graficar_historial(history, titulo=""):
    """Dibuja accuracy y loss de entrenamiento/validación."""
    h = history.history
    clave = "accuracy" if "accuracy" in h else "acc"
    epocas = range(1, len(h[clave]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    if titulo:
        fig.suptitle(titulo, fontsize=13)
    ax1.plot(epocas, h[clave], "r--", label="Entrenamiento")
    ax1.plot(epocas, h[f"val_{clave}"], "b", label="Validación")
    ax1.set_title("Precisión"); ax1.set_xlabel("Época"); ax1.set_ylabel("Accuracy")
    ax1.legend(); ax1.grid(True)
    ax2.plot(epocas, h["loss"], "r--", label="Entrenamiento")
    ax2.plot(epocas, h["val_loss"], "b", label="Validación")
    ax2.set_title("Pérdida"); ax2.set_xlabel("Época"); ax2.set_ylabel("Loss")
    ax2.legend(); ax2.grid(True)
    plt.tight_layout(); plt.show()
    print(f"Mejor val_accuracy: {max(h[f'val_{clave}']):.4f}")
    print(f"Mejor val_loss    : {min(h['val_loss']):.4f}")

print("Funciones auxiliares definidas: crear_callbacks, graficar_historial.")

## 3. Arquitecturas y estrategia de fine-tuning

Una función `build_<modelo>()` por arquitectura (idénticas a las de sus
notebooks originales) y una función `unfreeze_<modelo>(base)` que aplica la
estrategia de descongelado de la Fase 2 de cada uno.

In [ ]:
# ---------- VGG16 ----------
def build_vgg16():
    from tensorflow.keras.applications import VGG16
    from tensorflow.keras.applications.vgg16 import preprocess_input
    target_layer = "block5_conv3"   # última conv de VGG16 (14x14x512)
    inputs = Input(shape=(224, 224, 3))   # [0, 1] RGB
    # VGG16 preprocesado Caffe: RGB->BGR + resta de medias [103.939, 116.779, 123.68]
    x = layers.Lambda(lambda t: preprocess_input(t * 255.0))(inputs)
    base = VGG16(input_tensor=x, include_top=False, weights="imagenet")
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    modelo = Model(inputs, outputs, name="melanoma_vgg16")
    grad_cam_model = Model(inputs, [base.get_layer(target_layer).output, modelo.output])
    return modelo, base, grad_cam_model, target_layer

def unfreeze_vgg16(base):
    base.trainable = True
    for layer in base.layers[:-4]:
        layer.trainable = False


# ---------- ResNet50V2 ----------
def build_resnet50v2():
    from tensorflow.keras.applications import ResNet50V2
    target_layer = "post_relu"   # activación final de ResNet50V2 (7x7x2048)
    inputs = Input(shape=(224, 224, 3))            # [0, 1] RGB
    x = layers.Rescaling(2.0, offset=-1.0)(inputs)  # -> [-1, 1]
    base = ResNet50V2(input_tensor=x, include_top=False, weights="imagenet")
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    modelo = Model(inputs, outputs, name="melanoma_resnet50v2")
    grad_cam_model = Model(inputs, [base.get_layer(target_layer).output, modelo.output])
    return modelo, base, grad_cam_model, target_layer

def unfreeze_resnet50v2(base):
    base.trainable = True
    # Congelar TODAS las capas BatchNorm (estabilidad del fine-tuning)
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    # Descongelar las últimas ~80 capas no-BN
    for layer in base.layers[-80:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True


# ---------- EfficientNetV2S ----------
def build_efficientnetv2s():
    from tensorflow.keras.applications import EfficientNetV2S
    target_layer = "top_conv"   # última conv de EfficientNetV2S (7x7x1280)
    inputs = Input(shape=(224, 224, 3))     # [0, 1] RGB
    x = layers.Rescaling(255.0)(inputs)     # -> [0, 255] (preproc interno lo espera)
    base = EfficientNetV2S(input_tensor=x, include_top=False, weights="imagenet",
                           include_preprocessing=True)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    modelo = Model(inputs, outputs, name="melanoma_efficientnetv2s")
    grad_cam_model = Model(inputs, [base.get_layer(target_layer).output, modelo.output])
    return modelo, base, grad_cam_model, target_layer

def unfreeze_efficientnetv2s(base):
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    mid = len(base.layers) // 2
    for layer in base.layers[mid:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

print("Funciones de arquitectura definidas: build_/unfreeze_ para vgg16, resnet50v2, efficientnetv2s.")

In [ ]:
# Configuración de entrenamiento por arquitectura (fases 1 y 2 de cada notebook original)
MODEL_CONFIGS = [
    dict(
        id="vgg16", build=build_vgg16, unfreeze=unfreeze_vgg16,
        f1_lr=1e-4, f1_epochs=20, f1_es=7, f1_lr_pac=3,
        f2_opt="adam", f2_lr=1e-5, f2_epochs=30, f2_es=10, f2_lr_pac=4,
    ),
    dict(
        id="resnet50v2", build=build_resnet50v2, unfreeze=unfreeze_resnet50v2,
        f1_lr=1e-4, f1_epochs=20, f1_es=7, f1_lr_pac=3,
        f2_opt="adam", f2_lr=1e-6, f2_epochs=40, f2_es=12, f2_lr_pac=5,
    ),
    dict(
        id="efficientnetv2s", build=build_efficientnetv2s, unfreeze=unfreeze_efficientnetv2s,
        f1_lr=1e-4, f1_epochs=20, f1_es=7, f1_lr_pac=3,
        f2_opt="adam", f2_lr=1e-5, f2_epochs=40, f2_es=12, f2_lr_pac=5,
    ),
]
print("Modelos a entrenar, en orden:", [c["id"] for c in MODEL_CONFIGS])

## 4. Entrenamiento de los 3 modelos, uno tras otro

Para cada arquitectura: Fase 1 (backbone congelado, RMSprop) -> Fase 2
(fine-tuning con la estrategia de descongelado propia) -> evaluación en test
-> calibración (temperature scaling) -> guardado `.keras` + conversión a
TF.js. `tf.keras.backend.clear_session()` libera la GPU entre modelos.

In [ ]:
resultados = {}

for cfg in MODEL_CONFIGS:
    modelo_id = cfg["id"]
    print("\n" + "#" * 70)
    print(f"# ENTRENANDO: {modelo_id.upper()}")
    print("#" * 70)
    os.makedirs(os.path.join(MODEL_DIR, modelo_id), exist_ok=True)

    print(f"[{modelo_id}] Construyendo modelo (descarga de pesos ImageNet la primera vez, puede tardar)...")
    modelo, base, grad_cam_model, target_layer = cfg["build"]()
    print(f"[{modelo_id}] Modelo construido.")
    modelo.summary()
    ent = sum(int(tf.size(w)) for w in modelo.trainable_weights)
    tot = sum(int(tf.size(w)) for w in modelo.weights)
    print(f"Parámetros totales: {tot:,} | entrenables: {ent:,}")

    # --- Fase 1: extracción de características ---
    print(f"[{modelo_id}] Fase 1 — backbone congelado, RMSprop {cfg['f1_lr']} , {cfg['f1_epochs']} epochs")
    callbacks_f1 = crear_callbacks(modelo_id, "phase1.keras", cfg["f1_es"], cfg["f1_lr_pac"])
    modelo.compile(optimizer=RMSprop(learning_rate=cfg["f1_lr"]),
                   loss="binary_crossentropy", metrics=["accuracy"])
    history_f1 = modelo.fit(
        train_ds, validation_data=val_ds, epochs=cfg["f1_epochs"],
        class_weight=class_weight, callbacks=callbacks_f1, verbose=1,
    )
    graficar_historial(history_f1, f"{modelo_id} — Fase 1")

    # --- Fase 2: fine-tuning ---
    print(f"[{modelo_id}] Fase 2 — fine-tuning, Adam {cfg['f2_lr']} , {cfg['f2_epochs']} epochs")
    cfg["unfreeze"](base)
    callbacks_f2 = crear_callbacks(modelo_id, "finetuning.keras", cfg["f2_es"], cfg["f2_lr_pac"])
    modelo.compile(optimizer=Adam(learning_rate=cfg["f2_lr"]),
                   loss="binary_crossentropy", metrics=["accuracy"])
    history_f2 = modelo.fit(
        train_ds, validation_data=val_ds, epochs=cfg["f2_epochs"],
        class_weight=class_weight, callbacks=callbacks_f2, verbose=1,
    )
    graficar_historial(history_f2, f"{modelo_id} — Fine-tuning")

    # --- Evaluación en test ---
    print(f"[{modelo_id}] Evaluando en test...")
    test_loss, test_acc = modelo.evaluate(test_ds, verbose=1)
    y_true, y_pred_prob = [], []
    for imgs, labels in test_ds:
        p = modelo.predict(imgs, verbose=0).flatten()
        y_true.extend(labels.numpy().flatten())
        y_pred_prob.extend(p)
    y_true = np.array(y_true).astype(int)
    y_pred_prob = np.array(y_pred_prob)
    y_pred = (y_pred_prob >= 0.5).astype(int)

    print("\n" + "=" * 60)
    print(f"REPORTE DE CLASIFICACIÓN — {modelo_id} (umbral 0.5)")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensibilidad = tp / (tp + fn)
    especificidad = tn / (tn + fp)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.4f}")
    ax1.plot([0, 1], [0, 1], "k--", label="Azar")
    ax1.set_xlabel("Ratio Falso Positivo (FPR)")
    ax1.set_ylabel("Recall — Verdadero Positivo (TPR)")
    ax1.set_title(f"Curva ROC — {modelo_id}"); ax1.legend(); ax1.grid(True)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax2, cmap="Blues", colorbar=False)
    ax2.set_title(f"Matriz de confusión — {modelo_id}")
    plt.tight_layout(); plt.show()

    print(f"Sensibilidad : {sensibilidad:.3f}  ({tp}/{tp+fn} melanomas detectados)")
    print(f"Especificidad: {especificidad:.3f}  ({tn}/{tn+fp} benignos descartados)")
    print(f"Melanomas NO detectados (FN): {fn}  (el error más grave)")

    # --- Calibración (temperature scaling) ---
    print(f"[{modelo_id}] Calibrando (temperature scaling)...")
    val_logits, val_labels = [], []
    for imgs, labels in val_ds:
        p = modelo.predict(imgs, verbose=0).flatten()
        p = np.clip(p, 1e-7, 1 - 1e-7)
        val_logits.extend(np.log(p / (1 - p)))
        val_labels.extend(labels.numpy().flatten())
    val_logits = np.array(val_logits)
    val_labels = np.array(val_labels)

    def _nll(t, _logits=val_logits, _labels=val_labels):
        cal = 1 / (1 + np.exp(-_logits / t))
        return -np.mean(_labels * np.log(cal + 1e-7) + (1 - _labels) * np.log(1 - cal + 1e-7))

    T_opt = float(minimize_scalar(_nll, bounds=(0.05, 10), method="bounded").x)
    print(f"Temperatura óptima T = {T_opt:.4f}")

    # --- Guardado y conversión a TF.js ---
    ruta_keras = os.path.join(MODEL_DIR, modelo_id, f"melanoma_{modelo_id}_final.keras")
    modelo.save(ruta_keras)
    print("Modelo Keras guardado:", ruta_keras)

    try:
        import tensorflowjs as tfjs
        converted_dir = os.path.join(MODEL_DIR, modelo_id, "tfjs")
        os.makedirs(converted_dir, exist_ok=True)
        tfjs.converters.save_keras_model(
            modelo, converted_dir,
            quantization_dtype_map={"uint8": "*"},
            metadata={
                "temperature": T_opt,
                "version": "1.0.0",
                "modelId": modelo_id,
                "targetLayer": target_layer,
                "convertedAt": str(datetime.now()),
            },
        )
        print("TF.js guardado en:", converted_dir)
    except ImportError:
        print("tensorflowjs no disponible en este entorno; se omite la conversión.")

    resultados[modelo_id] = dict(
        test_acc=float(test_acc), test_loss=float(test_loss), auc=float(roc_auc),
        sensibilidad=float(sensibilidad), especificidad=float(especificidad),
        temperature=T_opt, tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
    )

    # Libera memoria de la GPU antes de construir el siguiente modelo
    del modelo, base, grad_cam_model
    tf.keras.backend.clear_session()
    print(f"[{modelo_id}] Terminado. Memoria liberada.")

print("\nEntrenamiento de los 3 modelos completado.")

## 5. Comparativa final

Resumen de los 3 modelos entrenados y guardado de `resultados` en JSON
(`MODEL_DIR/resultados_comparativa.json`) para copiarlo a
`demo/src/lib/constants.js`.

In [ ]:
print(f"{'Modelo':<18}{'Test Acc':>10}{'AUC':>10}{'Sensib.':>10}{'Especif.':>10}{'Temp.':>8}{'FN':>6}")
for modelo_id, r in resultados.items():
    print(f"{modelo_id:<18}{r['test_acc']:>10.4f}{r['auc']:>10.4f}"
          f"{r['sensibilidad']:>10.3f}{r['especificidad']:>10.3f}{r['temperature']:>8.3f}{r['fn']:>6}")

ruta_resultados = os.path.join(MODEL_DIR, "resultados_comparativa.json")
with open(ruta_resultados, "w") as f:
    json.dump(resultados, f, indent=2)
print("\nResultados guardados en:", ruta_resultados)

---
### Siguientes pasos

1. Descarga las carpetas `tfjs/` de `/kaggle/working/melanoma_model/<modelo_id>/`
   y copia su contenido a `demo/public/model/<modelo_id>/` en el repo.
2. Actualiza `temperature` (y las métricas AUC/sens/spec) de cada modelo en
   `demo/src/lib/constants.js` con los valores de `resultados_comparativa.json`.